In [ ]:
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score
import mlflow
from mlflow.models import infer_signature

In [ ]:
##write mlflow ui in the console in order to connect to http://127.0.0.1:5000

## set the tracking uri
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

In [ ]:
## load the dataset
X, y = datasets.load_iris(return_X_y=True)

# split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20)

# Define the model hyperparameters
params = {
    "penalty": "l2",
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

##train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

In [ ]:
## Prediction on the test set
y_pred = lr.predict(X_test)
y_pred

In [ ]:
# 🧮 Evaluate on the hold-out test set
accuracy = accuracy_score(y_test, y_pred)
print("Hold-out test accuracy:", accuracy)

# 🔁 Run 5-fold cross-validation on full dataset
cv_scores = cross_val_score(lr, X, y, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()
print(f"Cross-validated accuracy (mean ± std): {cv_mean:.3f} ± {cv_std:.3f}")

In [ ]:
### MLFLOW tracking
# This sets the URL where MLflow logs and UI are running.
# In this case, it's pointing to your local MLflow Tracking Server.
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")

## Create a new MLFLOW experiment
# Think of an experiment as a folder that contains multiple runs.
# You can group and compare all your training runs under this experiment name.
mlflow.set_experiment("MLFLOW Quickstart")

## Start an MLFLOW run
# This opens a logging context: everything inside (params, metrics, models)
# will be tracked and saved in MLflow for this specific run.
with mlflow.start_run():

    ## Log the hyperparameters
    # Logs all model configuration values (like solver, max_iter, etc.)
    # so you can later reproduce and compare runs.
    mlflow.log_params(params)

    ## Log evaluation metrics
    # Logs both hold-out accuracy and cross-validated accuracy
    # so you can monitor improvements across runs.
    mlflow.log_metric("holdout_accuracy", accuracy)
    mlflow.log_metric("cv_accuracy_mean", cv_mean)
    mlflow.log_metric("cv_accuracy_std", cv_std)

    ## Set a tag with additional metadata
    # Tags are useful for annotating runs (e.g., purpose, model type, notes)
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    ## Infer the model signature
    # This captures input/output schema — helps when serving the model later.
    signature = infer_signature(X_train, lr.predict(X_train))

    ## Log the model
    # Saves the trained model in MLflow format.
    # Also registers the model (optional) with a version-controlled name.
    # You’ll be able to track it, download it, and deploy it later.
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="iris_model",  # Folder name inside the run
        signature=signature,
        input_example=X_train,  # Optional: helps document usage
        registered_model_name="tracking-quickstart",  # Will appear in the Model Registry
    )

## 📦 MLflow Key Concepts Summary

### 🧠 Core Terms/ Priority (from most critical to optional)

| Priority  | Term                        | What it does                                                         | Why?                                                                              |
| --------- | --------------------------- | -------------------------------------------------------------------- | --------------------------------------------------------------------------------- |
| 🥇 Must   | **`set_experiment()`**      | Groups related runs under a named experiment (like a folder of runs) | Organises your runs; needed to track properly                                     |
| 🥈 High   | **`artifact_path`**         | Where the **model files are saved** *inside* the run (a subfolder)   | Required for saving the model                                                     |
| 🥉 Medium | **`registered_model_name`** | Adds the model to the **Model Registry** with a name and versioning  | Optional — but needed if you want **versioning** or **staging/production** models |
| 🧾 Nice   | **`set_tag()`**             | Adds custom metadata to a run (notes, model type, comments, etc.)    | Optional — for **human readability** (notes, model type, etc.)                    |


---

## 📁 Visual Diagram: How MLflow Organises Artifacts

Here’s a simplified diagram of how everything fits together when you log a model:

```
MLflow Tracking Server
│
└── Experiments/
    └── MLFLOW Quickstart (set_experiment)
        └── Run ID: abc123/
            ├── tags/                 ← set_tag (key-value metadata)
            ├── params/               ← log_params (hyperparameters)
            ├── metrics/              ← log_metric (accuracy, loss, etc.)
            ├── artifacts/
            │   └── iris_model/       ← artifact_path ("iris_model")
            │       ├── MLmodel       ← metadata + model format
            │       └── model.pkl     ← saved model file
            └── model registry/       ← (linked if registered_model_name is set)
```

---

In [ ]:
# 🔁 This is your second model in the same notebook
# We're testing a different solver: 'newton-cg'

from sklearn.linear_model import LogisticRegression

# 🧪 Define new model parameters
params_newton = {
    "solver": "newton-cg",  # Different solver
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 42,
}

# ⚙️ Train new model with newton-cg
lr_newton = LogisticRegression(**params_newton)
lr_newton.fit(X_train, y_train)

# 📈 Predict and evaluate
y_pred_newton = lr_newton.predict(X_test)
accuracy_newton = accuracy_score(y_test, y_pred_newton)

# 🔁 Run 5-fold cross-validation using full dataset
cv_scores = cross_val_score(lr_newton, X, y, cv=5)
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

# 🖨️ Output both metrics
print("Hold-out test accuracy:", accuracy_newton)
print("Cross-validated accuracy (mean ± std):", f"{cv_mean:.3f} ± {cv_std:.3f}")

# 🚀 Log this second model in the same experiment
# (you already called set_tracking_uri and set_experiment earlier)

with mlflow.start_run():  # New run for the second model

    # 📝 Log the second model's hyperparameters
    mlflow.log_params(params_newton)

    # 📊 Log both evaluation methods
    mlflow.log_metric("holdout_accuracy", accuracy_newton)
    mlflow.log_metric("cv_accuracy_mean", cv_mean)
    mlflow.log_metric("cv_accuracy_std", cv_std)

    # 🏷️ Add tags for traceability
    mlflow.set_tag("Solver", "newton-cg")
    mlflow.set_tag("Note", "Second model run using newton-cg")

    # 🔍 Infer input/output schema
    signature_newton = infer_signature(X_train, lr_newton.predict(X_train))

    # 💾 Save this model under a different artifact path and name
    model_info_newton = mlflow.sklearn.log_model(
        sk_model=lr_newton,
        artifact_path="iris_model_newtoncg",  # Different folder
        signature=signature_newton,
        input_example=X_train,
        registered_model_name="iris-lr-newtoncg",  # Unique model name
    )

## 🔍 Model Validation and Inference Using MLflow (PyFunc + Signature Check)

This cell demonstrates how to:

1. ✅ **Validate incoming payloads** before serving a model, using `mlflow.models.validate_serving_input`.  
2. 📥 **Load a saved model** from a previous MLflow run using `pyfunc.load_model`.  
3. 🔮 **Make predictions** on test data (`X_test`) using the loaded model.  
4. 📊 **Compare predictions vs actual values** in a well-formatted DataFrame for inspection or debugging.

This process helps ensure that the model is both deployable and still performing correctly after serialization.

In [ ]:
from mlflow.models import validate_serving_input
import json

# 📦 Get the URI (location) of the saved model artifact in MLflow (NOT REGISTERED MODEL)
# This URI points to the exact model logged in a previous run (from model_info),
# for example: runs:/<run_id>/iris_model
# If you want to test the second model you trained, replace with model_info_newton.model_uri.
model_uri = (
    model_info.model_uri
)  # You can also use model_info_newton.model_uri if it's your second model

# 🧪 Simulate a real-world prediction request by defining a JSON-like payload
# This is what a REST API would send to your model after deployment
# Each inner list is a sample with 4 features (just like the iris dataset)
serving_payload = """
{
  "inputs": [
    [5.7, 3.8, 1.7, 0.3],
    [6.3, 2.7, 4.9, 1.8]
  ]
}
"""

# ✅ Check if the provided inputs match the model's expected input signature
# This will raise an error if the payload structure is incorrect
validate_serving_input(model_uri, serving_payload)
print("✅ Input is valid and matches the model signature!")

# 📥 Load the model from MLflow’s storage location using the PyFunc interface
# This allows you to treat the model as a standard Python function
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

# 🔮 Make predictions on the real test set (X_test) using the loaded model
# This is useful to confirm that the saved model works after loading
predictions = loaded_model.predict(X_test)

# 🧾 Get the column names for the iris dataset to format the DataFrame nicely
iris_features_name = datasets.load_iris().feature_names

# 🧪 Create a DataFrame to compare model predictions with actual test labels
# This is helpful for debugging, reporting, or inspecting prediction quality
result = pd.DataFrame(X_test, columns=iris_features_name)
result["actual_class"] = y_test  # True labels
result["predicted_class"] = predictions  # Model predictions

In [ ]:
result

In [ ]:
## 📌 Why Register a Model in MLflow?

"""
Registering a model in MLflow means adding it to the **Model Registry**.

The Model Registry gives you:
- **Versioning** → Every time you update the model, it gets a new version (v1, v2, v3…).
- **Stages** → You can move models between "None", "Staging", and "Production" environments.
- **Centralised access** → Anyone in your team can load the latest staging or production-ready model.
- **Auditing** → You can see who promoted/demoted a model, and when.

📍 When should you register a model?
- After validating the model’s performance (e.g., on hold-out data or cross-validation)
- When you want to share it with your team or staging services
- Before deploying to staging or production
"""

# ⛔ WHY THIS CELL FAILS FOR YOU (HTTP 500 on transition):
# - You are likely running **`mlflow ui`** (UI-only). Stage transitions hit the **Model Registry REST API**,
#   which is available only when a full **tracking server** is running:
#   `mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns`
# - Even if your notebook points to http://127.0.0.1:5000, if that process is only `mlflow ui`,
#   calls like `transition_model_version_stage` to `/api/2.0/mlflow/model-versions/transition-stage`
#   will return 500.
# - Make sure the model was actually **registered** (you must log with `registered_model_name=...`)
#   **against the same server** and backend store.
# - Ensure there is at least one **model version** for the name you use (e.g., "tracking-quickstart").

# import mlflow
# from mlflow.tracking import MlflowClient
#
# print("Tracking URI:", mlflow.get_tracking_uri())  # sanity check
#
# client = MlflowClient()
# name = "tracking-quickstart"
#
# # Can be empty if the registry has no versions yet (or you registered against a different store).
# versions = client.get_latest_versions(name, stages=["None", "Staging", "Production"])
# latest_v = max(int(mv.version) for mv in versions)
#
# # ❗ This transition fails with 500 if only `mlflow ui` is running (no registry API backend).
# client.transition_model_version_stage(
#     name=name,
#     version=latest_v,
#     stage="Staging",
#     archive_existing_versions=True,
# )
#
# # Loading by stage also fails if no server/registry is running, or if no version is in "Staging".
# # staging_model = mlflow.pyfunc.load_model(model_uri=f"models:/{name}/Staging")
# # preds = staging_model.predict(X_test)

# ✅ HOW TO MAKE IT WORK:
# 1) Start a real tracking server (keep it running):
#    mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns
# 2) In the notebook, set:
#    mlflow.set_tracking_uri("http://127.0.0.1:5000")
# 3) Log a model with registration *against that server*:
#    mlflow.sklearn.log_model(..., registered_model_name="tracking-quickstart")
# 4) Then re-run:
#    - client.get_latest_versions(...)
#    - client.transition_model_version_stage(..., stage="Staging")
#    - mlflow.pyfunc.load_model("models:/tracking-quickstart/Staging")

In [ ]:
## 📦 Loading Registered Models Anywhere
"""
Once a model is in the registry, you can load it **by name and stage** (e.g., Production, Staging)
or **by version** (e.g., v1, v2), without worrying about artifact file paths.

This is extremely useful in production:
- Your API code can always point to the latest "Production" or "Staging" version.
- You can promote a new version without touching the API code.
- Works seamlessly across environments and team members.
"""

import mlflow
import mlflow.sklearn

# 📌 Option 1: Load by stage (recommended for production APIs)
# ⛔ COMMENTED OUT — WHY IT DOESN'T WORK RIGHT NOW:
# - You started `mlflow ui` (UI-only). Loading by "models:/.../Production" or "Staging"
#   hits the Model Registry REST API, which requires a real tracking server:
#   `mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlruns`
# - It also fails if there is **no version assigned** to the "Production" stage.
# - Ensure your notebook uses the same tracking URI as the server:
#   `mlflow.set_tracking_uri("http://127.0.0.1:5000")`
# production_model = mlflow.pyfunc.load_model("models:/tracking-quickstart/Production")
# prod_predictions = production_model.predict(X_test)
# print("Predictions from Production model:", prod_predictions[:5])

# ✅ Option 2: Load by explicit version (works without relying on stages)
# NOTE: Avoid "latest" unless you've created an alias named 'latest'.
#       Use a concrete version number (e.g., "6") or a stage like "Staging" once the server is set up.

model_name = "tracking-quickstart"
# Replace "6" below with the actual version you see in the registry UI
model_version = (
    "6"  # e.g., "1", "2", ...  (do NOT use "latest" unless you defined that alias)
)

model_uri = f"models:/{model_name}/{model_version}"
latest_model = mlflow.sklearn.load_model(model_uri)
latest_predictions = latest_model.predict(X_test)
print(f"Predictions from version {model_version}:", latest_predictions[:5])

# If you prefer stages later (after starting a real server and promoting a version):
# staging_model = mlflow.pyfunc.load_model(f"models:/{model_name}/Staging")
# preds = staging_model.predict(X_test)
# print("Predictions from Staging model:", preds[:5])